# Re-index Scriptures by Paragraph

Replaces page-level Qdrant points with **paragraph-level** points.
Uses OCR text already in Firestore — no re-OCR, no GPU.

## Payload per point
```
book_id, book_title, categories
page_number       — which page this paragraph is on
paragraph_number  — 1-based index within the page
gatha_number      — first gatha in this paragraph (if detected)
gatha_range       — range if multiple gathas span it
chapter_number    — carried forward from last chapter marker seen
preview           — first 400 chars of paragraph text (cleaned)
```

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "firebase-admin==6.3.0",
    "qdrant-client",
    "google-cloud-firestore",
    "requests",
], check=True)
print("✓ Dependencies ready")

In [ ]:
import os, json, re, time, uuid
import requests as _req
from google.oauth2 import service_account as _svc_acct
from google.cloud import firestore as _gfs
import firebase_admin
from firebase_admin import credentials
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, PointStruct

# ── Secrets ───────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _get(name): return _s.get_secret(name)
    print("✓ Secrets: Kaggle")
except Exception:
    try:
        from google.colab import userdata as _ud
        def _get(name): return _ud.get(name)
        print("✓ Secrets: Colab")
    except Exception:
        with open(os.path.expanduser("~/ask-aagam-secrets.json")) as _f:
            _ls = json.load(_f)
        def _get(name): return _ls[name]
        print("✓ Secrets: local file")

GCP_PROJECT = _get("GCP_PROJECT_ID")
sa_info     = json.loads(_get("GCP_SERVICE_ACCOUNT_JSON"))
HF_TOKEN    = _get("HF_TOKEN")

_sa_path = "/tmp/gcp_sa.json"
with open(_sa_path, "w") as _f: json.dump(sa_info, _f)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = _sa_path

if not firebase_admin._apps:
    firebase_admin.initialize_app(credentials.Certificate(sa_info), {"projectId": GCP_PROJECT})

_sa_creds = _svc_acct.Credentials.from_service_account_info(
    sa_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform",
            "https://www.googleapis.com/auth/datastore"],
)
db     = _gfs.Client(project=GCP_PROJECT, credentials=_sa_creds)
qdrant = QdrantClient(url=_get("QDRANT_URL"), api_key=_get("QDRANT_API_KEY"), timeout=60)

QDRANT_COLLECTION = "scripture_pages"
HF_API            = "https://router.huggingface.co/hf-inference/models/intfloat/multilingual-e5-large"
EMBED_BATCH_SIZE  = 8   # safe for HF free tier

print("✓ All clients connected")

In [ ]:

# ── Helpers ───────────────────────────────────────────────────────────────────

_DEVA = str.maketrans("०१२३४५६७८९", "0123456789")
def deva(s): return s.translate(_DEVA)

_DANDA = r"(?:॥|\|\||II|\|I|I\|)"
_NUM   = r"([०-९\d]+)"

GATHA_RE = [
    re.compile(rf"{_DANDA}\s*{_NUM}\s*{_DANDA}"),
    re.compile(r"(?:Gatha|गाथा|Verse|श्लोक)\s*[.:]?\s*([०-९\d]+)", re.IGNORECASE),
    re.compile(r"(?:^|\s)[\(\[]\s*([०-९\d]+)\s*[\)\]](?:\s|$)"),
]
CHAPTER_RE = [
    re.compile(r"अध्याय\s*[.:]?\s*([०-९\d]+)", re.IGNORECASE),
    re.compile(r"Chapter\s*[.:]?\s*([०-९\d]+)", re.IGNORECASE),
    re.compile(r"(?:प्रकरण|परिच्छेद)\s*[.:]?\s*([०-९\d]+)", re.IGNORECASE),
]

# ── Text cleaning ─────────────────────────────────────────────────────────────
# Preamble: "Here is the [corrected and] refined text...: [Corrected/Refined Text:]"
#           OR bare "Refined Text:" / "Corrected Text:" at start
_PREAMBLE = re.compile(
    r"^(?:"
    r"here is the (?:corrected and )?refined text[^:]*:\s*(?:(?:corrected|refined)\s+text:\s*)?"
    r"|(?:corrected|refined)\s+text:\s*"
    r")",
    re.I,
)
# AtmaDharma watermark — strips phrase, preserves content after "updares/updates"
_ATMADHARMA_WM = re.compile(
    r"(?:version\s+\S+:\s*)?remember\s+to\s+check\s+\S+.*?(?:updares?|updates?)\s*",
    re.I | re.S,
)
# Version prefix fallback: "Version X: " at line start
_VERSION_PFX = re.compile(r"^version\s+[^:\n]+:\s*", re.I | re.M)
_NOTES       = re.compile(r"\n?\*?\s*notes on (?:corrections|refinements)[\s\S]*", re.I)
_TRANS_BLOCK = re.compile(r"\n?\*?\s*transliteration[/\w\s]*:[\s\S]*", re.I)
_ENG_BLOCK   = re.compile(r"\n?\*?\s*english (?:interpretation|translation)[/\w\s]*:[\s\S]*", re.I)
_NOTE_PAREN  = re.compile(r"\(Note:[^)]*\)", re.I)
_STANZA_HDR  = re.compile(r"^Stanza\s+[\w०-९]+\s*\n?", re.M)
_URL         = re.compile(r"https?://\S+", re.I)
_SITE        = re.compile(r"(?i:www)\.\S+")
_PAGE_STAMP  = re.compile(r"^\d{1,4}\s+")

def clean(text: str) -> str:
    text = _PREAMBLE.sub("", text.strip())
    text = _ATMADHARMA_WM.sub("", text)
    text = _VERSION_PFX.sub("", text)
    text = _NOTES.sub("", text)
    text = _TRANS_BLOCK.sub("", text)
    text = _ENG_BLOCK.sub("", text)
    text = _NOTE_PAREN.sub("", text)
    text = _STANZA_HDR.sub("", text)
    text = _URL.sub("", text)
    text = _SITE.sub("", text)
    text = _PAGE_STAMP.sub("", text)
    return text.strip()

def gathas_in(text: str) -> list:
    found = set()
    for pat in GATHA_RE:
        for m in pat.finditer(text):
            try:
                n = int(deva(m.group(1)))
                if 1 <= n <= 9999: found.add(n)
            except ValueError: pass
    return sorted(found)

def chapter_in(text: str):
    for pat in CHAPTER_RE:
        m = pat.search(text)
        if m:
            try: return int(deva(m.group(1)))
            except ValueError: pass
    return None

def split_paragraphs(lines: list) -> list:
    """Split OCR lines into paragraph blocks (blank-line separated, min 3 words)."""
    paragraphs, current = [], []
    for line in lines:
        stripped = line.strip()
        if stripped:
            current.append(stripped)
        else:
            if current:
                text = " ".join(current)
                if len(text.split()) >= 3:
                    paragraphs.append(current)
                elif paragraphs:
                    paragraphs[-1].extend(current)
                current = []
    if current:
        text = " ".join(current)
        if len(text.split()) >= 3:
            paragraphs.append(current)
        elif paragraphs:
            paragraphs[-1].extend(current)
    return paragraphs if paragraphs else [lines]

def embed_batch(texts: list, retries: int = 5) -> list:
    """Embed via HF Inference API with exponential backoff."""
    prefixed = [f"passage: {t}" for t in texts]
    for attempt in range(retries):
        try:
            res = _req.post(
                HF_API,
                headers={"Authorization": f"Bearer {HF_TOKEN}", "Content-Type": "application/json"},
                json={"inputs": prefixed},
                timeout=60,
            )
            if res.status_code == 200:
                data = res.json()
                return data if isinstance(data[0], list) else [data]
            elif res.status_code in (503, 429):
                wait = 20 * (attempt + 1)
                print(f"    HF {res.status_code}, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise RuntimeError(f"HF embed failed ({res.status_code}): {res.text[:200]}")
        except _req.exceptions.Timeout:
            print(f"    Timeout attempt {attempt+1}, retrying...")
            time.sleep(10)
    raise RuntimeError("HF embedding failed after all retries")

# Smoke tests
t1 = clean("Here is the refined text based on your rules: Refined Text: सम्यक्\nNotes on Corrections Applied:\n1. Fixed fonts.")
assert t1 == "सम्यक्", repr(t1)

t2 = clean("Here is the corrected and refined text based on your rules: Corrected Text: Version OO२: remember To check hrtp / /wWW AtmaDharma com for updares मोक्ष")
assert t2 == "मोक्ष", repr(t2)

t3 = clean("Refined Text: content\nTransliteration/Meaning: abc")
assert t3 == "content", repr(t3)

t4 = clean("Version OO२: remember To check hrtp / /wWW AtmaDharma com for updares उठाया गया")
assert t4 == "उठाया गया", repr(t4)

print("✓ Helpers ready + smoke tests passed")


In [ ]:
# ── Fetch all ready scriptures ────────────────────────────────────────────────
scriptures = [
    {"id": d.id, **d.to_dict()}
    for d in db.collection("scriptures").stream()
    if d.to_dict().get("status") == "ready"
]
print(f"Found {len(scriptures)} ready scriptures:")
for s in scriptures:
    pc = s.get("pageCount", "?")
    print(f"  {s['id'][:8]}... — {s.get('title','?')} ({pc} pages)")

In [ ]:
# ── Main loop ─────────────────────────────────────────────────────────────────
grand_total_points = 0

for scripture in scriptures:
    book_id    = scripture["id"]
    book_title = scripture.get("title", book_id)
    categories = scripture.get("categories", [])

    print(f"\n{'═'*65}")
    print(f"  {book_title}")
    print(f"{'═'*65}")

    # 1. Delete old page-level points for this book
    print("  Deleting old points...")
    qdrant.delete(
        collection_name=QDRANT_COLLECTION,
        points_selector=Filter(must=[
            FieldCondition(key="book_id", match=MatchValue(value=book_id))
        ])
    )
    print("  ✓ Old points deleted")

    # 2. Fetch pages from Firestore
    pages_snap = (
        db.collection("scriptures").document(book_id)
          .collection("pages").order_by("pageNumber").stream()
    )
    pages = [(d.to_dict()["pageNumber"], d.to_dict()["lines"]) for d in pages_snap]
    print(f"  Fetched {len(pages)} pages")

    # 3–4. Build paragraph records with metadata
    records = []
    current_chapter = None

    for page_number, lines in pages:
        paras = split_paragraphs(lines)
        for para_idx, para_lines in enumerate(paras, start=1):
            raw_text  = " ".join(para_lines)
            text      = clean(raw_text)
            if not text: continue

            gs = gathas_in(text)
            ch = chapter_in(text)
            if ch: current_chapter = ch

            rec = {
                "text":             text,
                "page_number":      page_number,
                "paragraph_number": para_idx,
                "gatha_number":     str(gs[0]) if gs else None,
                "gatha_range":      f"{gs[0]}–{gs[-1]}" if len(gs) > 1 else (str(gs[0]) if gs else None),
                "chapter_number":   current_chapter,
            }
            records.append(rec)

    print(f"  {len(records)} paragraphs extracted")

    # 5. Embed in batches
    print(f"  Embedding via HF API (batch={EMBED_BATCH_SIZE})...")
    all_vectors = []
    for i in range(0, len(records), EMBED_BATCH_SIZE):
        batch_texts = [r["text"] for r in records[i:i + EMBED_BATCH_SIZE]]
        vecs = embed_batch(batch_texts)
        all_vectors.extend(vecs)
        if (i // EMBED_BATCH_SIZE) % 10 == 0:
            print(f"    {len(all_vectors)}/{len(records)} embedded")
        time.sleep(0.3)

    # 6. Build and upsert Qdrant points
    points = []
    for idx, rec in enumerate(records):
        payload = {
            "book_id":          book_id,
            "book_title":       book_title,
            "categories":       categories,
            "page_number":      rec["page_number"],
            "paragraph_number": rec["paragraph_number"],
            "preview":          rec["text"][:400],
        }
        if rec["gatha_number"]:   payload["gatha_number"]  = rec["gatha_number"]
        if rec["gatha_range"]:    payload["gatha_range"]   = rec["gatha_range"]
        if rec["chapter_number"]: payload["chapter_number"] = rec["chapter_number"]

        points.append(PointStruct(
            id=str(uuid.uuid4()),
            vector=all_vectors[idx],
            payload=payload,
        ))

    UPSERT_CHUNK = 100
    for i in range(0, len(points), UPSERT_CHUNK):
        qdrant.upsert(collection_name=QDRANT_COLLECTION, points=points[i:i + UPSERT_CHUNK])

    grand_total_points += len(points)
    print(f"  ✓ {len(points)} paragraph-level points upserted")

print(f"\n{'═'*65}")
print(f"  DONE — {grand_total_points} total paragraph points in Qdrant")
print(f"{'═'*65}")

In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────
total = qdrant.count(collection_name=QDRANT_COLLECTION).count
print(f"Total Qdrant points: {total}")

pts, _ = qdrant.scroll(collection_name=QDRANT_COLLECTION, limit=10, with_payload=True)
print("\nSample points:")
for pt in pts:
    pl = pt.payload
    loc = f"Gatha {pl.get('gatha_range')}" if pl.get('gatha_range') else f"Page {pl['page_number']}"
    ch  = f" Ch.{pl['chapter_number']}" if pl.get('chapter_number') else ""
    print(f"  [{pl['book_title'][:30]}]{ch} {loc} Para {pl['paragraph_number']}")
    print(f"    {pl.get('preview','')[:80]}...")

# Per-book counts
print("\nPoints per book:")
for scripture in scriptures:
    c = qdrant.count(
        collection_name=QDRANT_COLLECTION,
        count_filter=Filter(must=[FieldCondition(key="book_id", match=MatchValue(value=scripture["id"]))]),
    ).count
    print(f"  {scripture.get('title','?')[:40]}: {c} paragraphs")

In [ ]:
# ── Re-run ONLY मोक्षमार्गप्रकाशक with fixed clean() ──────────────────────────
TARGET_TITLE = "मोक्षमार्गप्रकाशक"
target = next((s for s in scriptures if s.get("title") == TARGET_TITLE), None)
assert target, f"Not found: {TARGET_TITLE}"

book_id    = target["id"]
book_title = target.get("title", book_id)
categories = target.get("categories", [])

print(f"Re-indexing: {book_title}")

qdrant.delete(
    collection_name=QDRANT_COLLECTION,
    points_selector=Filter(must=[FieldCondition(key="book_id", match=MatchValue(value=book_id))])
)
print("  ✓ Old points deleted")

pages_snap = (
    db.collection("scriptures").document(book_id)
      .collection("pages").order_by("pageNumber").stream()
)
pages = [(d.to_dict()["pageNumber"], d.to_dict()["lines"]) for d in pages_snap]
print(f"  Fetched {len(pages)} pages")

records = []
current_chapter = None
for page_number, lines in pages:
    paras = split_paragraphs(lines)
    for para_idx, para_lines in enumerate(paras, start=1):
        text = clean(" ".join(para_lines))
        if not text: continue
        gs = gathas_in(text)
        ch = chapter_in(text)
        if ch: current_chapter = ch
        records.append({
            "text": text, "page_number": page_number,
            "paragraph_number": para_idx,
            "gatha_number": str(gs[0]) if gs else None,
            "gatha_range": f"{gs[0]}–{gs[-1]}" if len(gs) > 1 else (str(gs[0]) if gs else None),
            "chapter_number": current_chapter,
        })

print(f"  {len(records)} paragraphs extracted")

all_vectors = []
for i in range(0, len(records), EMBED_BATCH_SIZE):
    vecs = embed_batch([r["text"] for r in records[i:i + EMBED_BATCH_SIZE]])
    all_vectors.extend(vecs)
    if (i // EMBED_BATCH_SIZE) % 10 == 0:
        print(f"    {len(all_vectors)}/{len(records)} embedded")
    time.sleep(0.3)

points = []
for idx, rec in enumerate(records):
    payload = {
        "book_id": book_id, "book_title": book_title, "categories": categories,
        "page_number": rec["page_number"], "paragraph_number": rec["paragraph_number"],
        "preview": rec["text"][:400],
    }
    if rec["gatha_number"]:   payload["gatha_number"]   = rec["gatha_number"]
    if rec["gatha_range"]:    payload["gatha_range"]    = rec["gatha_range"]
    if rec["chapter_number"]: payload["chapter_number"] = rec["chapter_number"]
    points.append(PointStruct(id=str(uuid.uuid4()), vector=all_vectors[idx], payload=payload))

for i in range(0, len(points), 100):
    qdrant.upsert(collection_name=QDRANT_COLLECTION, points=points[i:i+100])

print(f"  ✓ {len(points)} paragraph-level points upserted for {book_title}")
